# التنبؤ بالموضع النهائي لـ PUBG



![](https://github.com/4ku/PUBG-prediction/raw/master/pictures/banner.png)



## 1. شرح الميزات والبيانات



في البداية أخبرنا شيئًا عن اللعبة. **PlayerUnknown's Battlegrounds (PUBG)** - لعبة معركة ملكية متعددة اللاعبين عبر الإنترنت.  يتم إسقاط ما يصل إلى 100 لاعب على الجزيرة خالي الوفاض، ويجب عليهم استكشاف اللاعبين الآخرين والبحث عنهم ونهبهم والقضاء عليهم حتى يتبقى لاعب واحد فقط واقفًا، وكل ذلك بينما تستمر منطقة اللعب في التقلص.  <br/>
لقد اجتاحت ألعاب الفيديو بأسلوب Battle Royale العالم. لذلك أصبحت لعبة PUBG ذات شعبية كبيرة. مع بيع أكثر من 50 مليون نسخة، فهي خامس أفضل لعبة مبيعًا على الإطلاق، ولديها الملايين من اللاعبين النشطين شهريًا.<br/>



<img src="@@KEEP_00050@@ width="1000" height="600"> 



<img src="@@KEEP_00051@@ width="1000" height="600"> 


**المهمة**: استخدام إحصائية اللاعب أثناء المباراة، توقع المركز النهائي لهذا اللاعب، حيث 0 هو المركز الأخير و1 هو الفائز، عشاء الدجاج. 
<br/><br/>
تحتوي مجموعة البيانات على ما يزيد عن 65000 لعبة من بيانات اللاعبين مجهولة المصدر، والتي يمكنك تنزيلها من موقع [kaggle](https://www.kaggle.com/c/pubg-finish-placement-prediction/data). كل صف من البيانات هو إحصائيات اللاعب في نهاية المباراة.  <br/> 
تأتي البيانات من جميع أنواع المباريات: المعزوفات المنفردة والثنائية والفرق والمخصصة؛ ليس هناك ما يضمن وجود 100 لاعب في المباراة الواحدة، ولا 4 لاعبين على الأكثر في كل مجموعة. <br/>
يمكن أن تكون الإحصائيات مثل - عدد مرات قتل اللاعب، ومطابقته، وهويته الجماعية والشخصية، ومقدار المسافة التي قطعها، وما إلى ذلك...
<br/> **WinPlacePerc** - هي ميزة مستهدفة على مقياس من 1 (المركز الأول) إلى 0 (المركز الأخير) - المركز المئوي للفوز.
 <br/> <br/>
يمكن أن يكون حل المهمة مفيدًا للاعبي PUBG لفهم المعلمات المهمة والتكتيك الذي يجب اختياره. باستخدام [PUBG Developer API](https://developer.pubg.com/) يمكننا أيضًا جمع البيانات الخاصة بنا مع المزيد من الميزات. لذلك، من الممكن إنشاء الكثير من التطبيقات المختلفة التي ستساعد اللاعبين. على سبيل المثال، التطبيق به مساعد شخصي، والذي سيقدم لك النصائح، والمهارة التي يجب عليك تدريبها. 
 <br/> <br/>
دعونا ننظر إلى البيانات



##2-3 تحليل البيانات الأولية وتحليل البيانات المرئية


In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import scipy.stats as sc
import gc
import warnings

In [ ]:
plt.rcParams['figure.figsize'] = 15,8
sns.set(rc={'figure.figsize':(15,8)})
pd.options.display.float_format = '{:.2f}'.format
warnings.filterwarnings('ignore')
gc.enable()

In [ ]:
train = pd.read_csv('../input/train_V2.csv')
test = pd.read_csv('../input/test_V2.csv')
train.head()


### حقول البيانات* **DBNOs** - عدد اللاعبين الأعداء الذين تم طردهم.
* **التمريرات الحاسمة** - عدد اللاعبين الأعداء الذين ألحق بهم هذا اللاعب الضرر والذين قُتلوا على يد زملائه في الفريق.
* **التعزيزات** - عدد عناصر التعزيز المستخدمة.
* **damageDealt** - إجمالي الضرر الناتج. ملحوظة: يتم طرح الضرر الذي يلحق بالنفس.
* **headshotKills** - عدد لاعبي العدو الذين قتلوا بطلقات في الرأس.
* **الشفاء** - عدد عناصر الشفاء المستخدمة.
* **المعرف** - معرف اللاعب
* **killPlace** - الترتيب في المباراة لعدد لاعبي العدو الذين قتلوا.
* **killPoints** - تصنيف خارجي للاعب يعتمد على عمليات القتل. (فكر في هذا باعتباره تصنيف Elo حيث تكون عمليات القتل فقط مهمة.) إذا كانت هناك * قيمة أخرى غير -1 في نقاط التصنيف، فيجب معاملة أي 0 في نقاط القتل على أنها "لا شيء".
* **killStreaks** - أقصى عدد من لاعبي العدو الذين يتم قتلهم في فترة زمنية قصيرة.
* **القتلى** - عدد القتلى من لاعبي العدو.
* **أطول عملية قتل** - أطول مسافة بين لاعب وآخر تم قتله وقت الوفاة. قد يكون هذا مضللاً، حيث أن إسقاط اللاعب والابتعاد عنه قد يؤدي إلى إحصائيات كبيرة لأطول عملية قتل.
* **matchDuration** - مدة المباراة بالثواني.
* **معرف المباراة** - معرف لتحديد المطابقة. لا توجد مطابقات في كل من مجموعة التدريب والاختبار.
* **matchType** - سلسلة تحدد وضع اللعبة الذي تأتي منه البيانات. الأوضاع القياسية هي "solo"، و"duo"، و"squad"، و"solo-fpp"، و"duo-fpp"، و"squad-fpp"؛ الأوضاع الأخرى هي من الأحداث أو التطابقات المخصصة.
* **rankPoints** - تصنيف اللاعب الشبيه بـ Elo. هذا التصنيف غير متسق وتم إهماله في الإصدار التالي من واجهة برمجة التطبيقات، لذا استخدمه بحذر. قيمة -1 تحل محل "لا شيء".
* **الإحياء** - عدد المرات التي قام فيها هذا اللاعب بإحياء زملائه في الفريق.
* **rideDistance** - إجمالي المسافة المقطوعة في المركبات مقاسة بالأمتار.
* **حالات القتل على الطريق** - عدد حالات القتل أثناء وجودك في السيارة.
* **swimDistance** - إجمالي المسافة المقطوعةعن طريق السباحة مقاسة بالأمتار.
* **teamKills** - عدد المرات التي قتل فيها هذا اللاعب زميله في الفريق.
* **مركبة مدمرة** - عدد المركبات المدمرة.
* **walkDistance** - إجمالي المسافة المقطوعة سيرًا على الأقدام مقاسة بالأمتار.
* **الأسلحة التي تم الحصول عليها** - عدد الأسلحة التي تم الحصول عليها.
* **winPoints** - تصنيف خارجي للاعب يعتمد على الفوز. (فكر في هذا كتصنيف Elo حيث الفوز فقط هو المهم.) إذا كانت هناك قيمة غير -1 في rankPoints، فيجب معاملة أي 0 في winPoints على أنها "لا شيء".
* **groupId** - معرف لتحديد مجموعة داخل المباراة. إذا لعبت نفس المجموعة من اللاعبين في مباريات مختلفة، فسيكون لديهم معرف مجموعة مختلف في كل مرة.
* **numGroups** - عدد المجموعات التي لدينا بيانات عنها في المباراة.
* **maxPlace** - أسوأ مركز لدينا بيانات عنه في المباراة. قد لا يتطابق هذا مع numGroups، حيث تتخطى البيانات أحيانًا المواضع.
* **winPlacePerc** - هدف التنبؤ. هذا هو المركز المئوي للفوز، حيث 1 يتوافق مع المركز الأول، و0 يتوافق مع المركز الأخير في المباراة. يتم حسابه من maxPlace، وليس من numGroups، لذلك من الممكن أن تكون هناك أجزاء مفقودة في المباراة


In [ ]:
train.info()


لدينا 4.5 مليون سجل إحصائيات اللاعب! <br/>
<br/>
تحقق الآن من مجموعة البيانات بحثًا عن القيم المفقودة


In [ ]:
display(train[train.isnull().any(1)])
display(test[test.isnull().any(1)])


يوجد صف واحد فقط بقيمة nan، لذلك دعونا نسقطه


In [ ]:
train.drop(2744604, inplace=True)


معلومات عامة عن كل عمود


In [ ]:
train.describe()


يمكننا بالفعل أن نخمن أن الميزة المستهدفة لها توزيع موحد. ذلك لأن ميزة winPlacePerc تم تحجيمها بالفعل وبعد كل مباراة يمكن للاعب الحصول على مكان واحد فقط.


In [ ]:
train['winPlacePerc'].hist(bins=25);

يمكننا أن نلاحظ أن 0 والقيم أكثر من غيرها.  ذلك لأن المركز الأول والأخير موجود في كل مباراة) <br/>
من الواضح أن WinPlacePerc لديه توزيع موحد، ولكن دعونا نتحقق من الميزة المستهدفة للتأكد من الحالة الطبيعية والتواء التوزيع (بسبب المهمة) 


In [ ]:
print(sc.normaltest(train['winPlacePerc']))
print('Skew: ', sc.skew(train['winPlacePerc']))


قيمة P هي صفر، لذا فإن هذا التوزيع غير طبيعي <br/>
الانحراف قريب من الصفر، وبالتالي فإن التوزيع متماثل تقريبًا



انظر الآن إلى توزيع الميزات ذات الحد الأعلى (للتخلص من القيم المتطرفة) وبدون قيم صفرية (بسبب وجود الكثير من القيم الصفرية)
<br/> قم أيضًا بإنشاء boxplots لرؤية ميزة هدف الارتباط من قيم الميزات


In [ ]:
def featStat(featureName, constrain,plotType):
    feat = train[featureName][train[featureName]>0]
    data = train[[featureName,'winPlacePerc']].copy()
    q99 = int(data[featureName].quantile(0.99))
    plt.rcParams['figure.figsize'] = 15,5;   
    
    if constrain!=None:
        feat = feat[feat<constrain]
    if plotType == 'hist':
        plt.subplot(1,2,1)
        feat.hist(bins=50);
        plt.title(featureName);
        
        n = 20
        cut_range = np.linspace(0,q99,n)
        cut_range = np.append(cut_range, data[featureName].max())
        data[featureName] = pd.cut(data[featureName],
                                         cut_range,
                                         labels=["{:.0f}-{:.0f}".format(a_, b_) for a_, b_ in zip(cut_range[:n], cut_range[1:])],
                                         include_lowest=True
                                        )
        ax = plt.subplot(1,2,2)
        sns.boxplot(x="winPlacePerc", y=featureName, data=data, ax=ax, color="#2196F3")
        ax.set_xlabel('winPlacePerc', size=14, color="#263238")
        ax.set_ylabel(featureName, size=14, color="#263238")
        plt.gca().xaxis.grid(True)
        plt.tight_layout()
           
    if plotType == 'count':        
        plt.subplot(1,2,1)
        sns.countplot(feat, color="#2196F3");
        
        plt.subplot(1,2,2)
        data.loc[data[featureName] > q99, featureName] = q99+1
        x_order = data.groupby(featureName).mean().reset_index()[featureName]
        x_order.iloc[-1] = str(q99+1)+"+"
        data[featureName][data[featureName] == q99+1] = str(q99+1)+"+"
        
        ax = sns.boxplot(x=featureName, y='winPlacePerc', data=data, color="#2196F3", order = x_order);
        ax.set_xlabel(featureName, size=14, color="#263238")
        ax.set_ylabel('WinPlacePerc', size=14, color="#263238")
    plt.tight_layout()


**قتلى وأضرار**



<img src="@@KEEP_00054@@ width="600" height="400"> 


In [ ]:
featStat('kills',15,'count');
plt.show();
featStat('longestKill',400,'hist');
plt.show();
featStat('damageDealt',1000,'hist');


**شفاء وتعزيز**



<img src="@@KEEP_00055@@ width="800" height="600"> 



<img src="@@KEEP_00056@@ width="600" height="400"> 


In [ ]:
featStat('heals',20,'count')
plt.show()
featStat('boosts',12,'count')


** المسافة **



<img src="@@KEEP_00057@@ width="1200" height="800"> 



<img src="@@KEEP_00058@@ width="600" height="400"> 


In [ ]:
featStat('walkDistance',5000,'hist')
plt.show()
featStat('swimDistance',500,'hist')
plt.show()
featStat('rideDistance',12000,'hist')


** بعض الميزات الأخرى **



<img src="@@KEEP_00059@@ width="600" height="400"> 


In [ ]:
featStat('weaponsAcquired',15,'count')
plt.show()
featStat('vehicleDestroys',None,'count')

In [ ]:
features = ['kills', 'longestKill', 'damageDealt', 'heals', 'boosts', 'walkDistance', 'swimDistance', 'rideDistance', 'weaponsAcquired', 'vehicleDestroys']
zeroPerc = ((train[features] == 0).sum(0) / len(train)*100).sort_values(ascending = False)
sns.barplot(x=zeroPerc.index , y=zeroPerc, color="#2196F3");
plt.title("Percentage of zero values")
plt.tight_layout()


كما نرى، مع زيادة قيمة هذه الميزات، تزداد أيضًا احتمالية الفوز. لذا فإن الميزات الموضحة أعلاه ترتبط بشكل جيد بالميزة المستهدفة. <br/>
رسم الميزات المتبقية


In [ ]:
df = train.drop(columns=['Id','matchId','groupId','matchType']+features)
df[(df>0) & (df<=df.quantile(0.99))].hist(bins=25,layout=(5,5),figsize=(15,15));
plt.tight_layout()


### ارتباطات الميزات 


In [ ]:
f,ax = plt.subplots(figsize=(15, 13))
sns.heatmap(df.corr(), annot=True, fmt= '.1f',ax=ax,cbar=False)
plt.show()


خذ الميزات الأكثر ارتباطًا بالميزة المستهدفة


In [ ]:
f,ax = plt.subplots(figsize=(11, 11))
cols = abs(train.corr()).nlargest(6, 'winPlacePerc')['winPlacePerc'].index
hm = sns.heatmap(np.corrcoef(train[cols].values.T), annot=True, square=True, fmt='.2f',  yticklabels=cols.values, xticklabels=cols.values)
print(", ".join(cols[1:]), " most correlate with target feature")
plt.show()


دعونا نجعل المخططات الزوجية. يمكننا أن نرى بوضوح الارتباط مع winPlacePerc (ولكن من الصعب رؤية الأسلحة التي تم الحصول عليها فقط)


In [ ]:
sns.set(font_scale=2)
sns.pairplot(train, y_vars=["winPlacePerc"], x_vars=cols[1:],height=8);
sns.set(font_scale=1)


### إحصائيات المباراة


In [ ]:
print("Number of match in train dataset:",train['matchId'].nunique())

In [ ]:
playersJoined = train.groupby('matchId')['matchId'].transform('count')
sns.countplot(playersJoined[playersJoined>=75])
plt.title('playersJoined');

In [ ]:
ngroupsByMatch = train.groupby('matchId')['groupId'].nunique()
ax = sns.countplot(ngroupsByMatch)
plt.title('Number of groups by match');
ax.xaxis.set_major_formatter(ticker.FormatStrFormatter('%d'))
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=5)) #Starts from 0 not from 1:(

In [ ]:
train.matchDuration.hist(bins=50);


يمكننا أن نرى 3 قمم في المخطط الثاني وقمتين في مخطط مدة المباراة. من المفترض أن ذلك يعتمد على نوع المطابقة.



** بعض الإحصائيات حسب نوع المباراة **


In [ ]:
plt.rcParams['figure.figsize'] = 18,7;
types = train.groupby('matchType').size().sort_values(ascending=False)
sns.barplot(x=types.index,y=types.values);
plt.title("Number of players by matchType");
plt.tight_layout()

لذلك، عادة ما يلعب الناس في فرق أو أزواج. (أو ربما مجرد بيانات تم جمعها بهذه الطريقة) <br/><br/>
وفي النهاية بعض الأرقام التي تصف كل نوع من الألعاب حسب عدد اللاعبين والمجموعات والمباريات وغيرها.
<br/> في هذا الجدول np.size - عدد اللاعبين و_num - عدد المباريات. يمكننا أن نرى أن maxPlace وnumGroups متماثلان تقريبًا.


In [ ]:
def _min(x):
    return x.value_counts().values.min()
def _max(x):
    return x.value_counts().values.max()
def _avg(x):
    return x.value_counts().values.mean()
def _med(x):
    return np.median(x.value_counts().values)
def _num(x):
    return x.nunique()
infostat = train.groupby('matchType').agg({
    "matchId": [np.size, _num, _min,_med,_max], #np.size - number of players, _num - number of matches
    "groupId": [_min,_med,_max],
    "matchDuration": [min,np.median, max], 
    "maxPlace": [min,np.median,max],
    "numGroups":[min,np.median,max]
    }).sort_values(by=('matchId','size'),ascending=False)  
display(infostat)


## 4. الرؤى والتبعيات الموجودة



      لقد وجدنا أن walkDistance، وkillPlace، والتعزيزات، والأسلحة المكتسبة، والضرر الذي تم التعامل معه - هي الميزات الأكثر ارتباطًا. من السهل تخمين السبب.
<br/>      إذا كنت قريبًا من القمة، فمن المرجح أن تمشي لمسافة أكبر، لأنه يجب أن تكون في الدائرة (منطقة اللعبة). على الأرجح، أن تجد سلاحًا جيدًا و/أو تقتل شخصًا ما. إذا قتلت شخصًا ما، فيمكن لعدوك أن يؤذيك، لذا فمن الأفضل استخدام التعزيز. بالقرب من كل عدو مقتول، يمكنك العثور على غنائمه، ومن المحتمل أن تحصل على بعض أسلحته.
<br/>
      يمكننا أيضًا أن نرى أن الكثير من الأشخاص يلعبون في فرق أو ثنائي (يلعبون في مجموعات). اللاعبون في فريق واحد لديهم نفس المركز النهائي. النتيجة النهائية تعتمد على العمل الجماعي. لذا من الأفضل رؤية الإحصائيات العامة حسب الفريق، وليس حسب اللاعب المنفصل.



*مناطق اللعب*
![مناطق اللعبة](https://github.com/4ku/PUBG-prediction/raw/master/pictures/Circle%20zones.png)



## 5. اختيار المقاييس


   هذه المهمة هي مشكلة الانحدار. بالنسبة لمشكلة الانحدار، فإننا نعرف فقط `mean absolute error` (MAE)، `mean squared error` (MSE؛ يوجد أيضًا جذر MSE) و`mean squared log error` (MSLE؛ جذر MSLE موجود أيضًا). <br/>
   هدفنا له توزيع موحد بمدى من 0 إلى 1. لكن MSLE أكثر ملاءمة للتوزيع غير الموحد وعادةً ما يستخدم MSE، عندما تكون الأخطاء الكبيرة غير مرغوب فيها بشكل خاص. نحن لسنا في هذه الحالة، لذا فإن **MAE** سيكون مناسبًا لنا.



![](https://github.com/4ku/PUBG-prediction/raw/master/pictures/Love%20MAE.png)



## 6. اختيار النموذج



قررت استخدام **LightGBM**. مع LightGBM، يكون من السهل العمل مع مجموعات البيانات الكبيرة (حالتنا). LightGBM أسرع من XGBoost على سبيل المثال ويعطي نقاطًا جيدة في نفس الوقت. هناك الكثير من المعلمات التي يجب ضبطها (المشكلة الرئيسية) وهي تدعم حل مشكلات الانحدار.



<img src="@@KEEP_00062@@ width="500" height="190"> 


In [ ]:
import lightgbm as lgb


## 7. المعالجة المسبقة للبيانات



كما ذكرت سابقًا، سنقوم بتجميع إحصائيات اللاعبين إلى فرق (حسب معرف المجموعة).


In [ ]:
# Function, which reduce memory usage. 
# This function I took from ready kernel (https://www.kaggle.com/gemartin/load-data-reduce-memory-usage)
def reduce_mem_usage(df):
    """ iterate through all the columns of a dataframe and modify the data type
        to reduce memory usage.        
    """
    start_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage of dataframe is {:.2f} MB'.format(start_mem))

    for col in df.columns:
        col_type = df[col].dtype

        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)

    end_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage after optimization is: {:.2f} MB'.format(end_mem))
    print('Decreased by {:.1f}%'.format(100 * (start_mem - end_mem) / start_mem))
    return df


     في الخطوات التالية سنقوم بإنشاء ميزات جديدة. ولهذا السبب سوف تتكرر هذه الخطوة مرة أخرى. لذلك، في هذه المرحلة، قم بإعداد بيانات بسيطة. نحن فقط نجمع الجميع حسب الفريق ثم نرتب الترتيب في كل مباراة.
<br/>
 >  الترتيب - القياس، حيث يتم استبدال أدنى قيمة في الجدول الأولي بقيمة حوالي صفر (يعتمد على التوزيع؛ لا تقل عن 0) واستبدال الحد الأقصى للقيمة إلى قيمة حوالي 1 (ليس أعلى من 1).


In [ ]:
def initial_preparing(df, Debug):
    if Debug:
        df = df[df['matchId'].isin(df['matchId'].unique()[:2000])]
    # Drop next columns. *Points features don't correlate with target feature, need
    # more EDA to understand how they work.
    df.drop(columns=['killPoints','rankPoints','winPoints','matchType','maxPlace','Id'],inplace=True)
    X = df.groupby(['matchId','groupId']).agg(np.mean)
    X = reduce_mem_usage(X)
    y = X['winPlacePerc']     
    X.drop(columns=['winPlacePerc'],inplace=True)
    X_ranked = X.groupby('matchId').rank(pct=True)
    X = X.reset_index()[['matchId','groupId']].merge(X_ranked, how='left', on=['matchId', 'groupId'] )
    X.drop(['matchId','groupId'],axis=1, inplace=True)
    X = reduce_mem_usage(X)
    return X, y

In [ ]:
X_train, y = initial_preparing(train.copy(),False)


قم بتقسيم مجموعة بيانات القطار الخاصة بنا إلى جزء سنقوم بتدريبه (X_train؛ نفس الاسم)، وجزء سنتحقق من الخطأ به.


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_holdout, y_train, y_holdout = train_test_split(X_train, y, test_size=0.2, random_state=666)


## 8-9. التحقق من الصحة وتعديل المعلمات الفائقة للنموذج. إنشاء ميزات جديدة ووصف هذه العملية


In [ ]:
from sklearn.model_selection import cross_val_score
import sklearn.metrics
from sklearn.model_selection import GridSearchCV

اخترت 5 طيات في التحقق المتبادل. لدينا مجموعة بيانات كبيرة، لذا ليس من الضروري تعيين المزيد من الطيات، 80% لجزء التدريب يكفي لتدريب النموذج.  علاوة على ذلك، إذا اخترت رقمًا أعلى، فسيستغرق حسابه الكثير من الوقت. 


In [ ]:
%%time
lgtrain = lgb.Dataset(X_train, label=y_train.reset_index(drop=True))
res = lgb.cv({'metric': 'mae'},lgtrain, nfold=5,stratified=False,seed=666)
print("Mean score:",res['l1-mean'][-1])
gc.collect()


إذن، درجاتنا هي 0.0644. انها ليست سيئة. هذا يعني أن خطأ النموذج الخاص بنا هو +-6.42 مواضع (إذا كان هناك 100 لاعب على الخادم)



دعونا نضيف ميزات جديدة ونقوم بالترتيب مرة أخرى.



عندما نقوم بتجميع مجموعة البيانات حسب معرف المجموعة، فإننا ننشئ ميزات "جديدة"، أي يمكننا التجميع بطرق مختلفة. <br/>
على سبيل المثال، "التعزيزات": مجموع - العدد الإجمالي لاستخدام التعزيزات في فريق واحد.


In [ ]:
team_features = {
        'assists': [sum, np.mean, np.size], #np.size - size of team
        'boosts' : [sum, np.var, np.mean], 
        'heals': [sum, np.var, np.mean],
        'damageDealt': [np.var,min,max,np.mean],
        'DBNOs': [np.var,max,np.mean],
        'headshotKills': [max,np.mean],
        'killPlaceScall':[sum, min,max, np.var, np.mean],
        'kills': [ sum, max, np.var,np.mean],
        'killStreaks': [max,np.var,np.mean],
        'longestKill': [max, np.mean, np.var],
        'revives': sum,
        'rideDistance': [sum, np.mean,np.var],
        'swimDistance': [np.var],
        'teamKills': sum,
        'vehicleDestroys': sum,
        'walkDistance': [np.var,np.mean],
        'weaponsAcquired': [np.mean],
        'damageRate': [np.var,min,max,np.mean],
        'headshotRate': [np.var,max,np.mean],
        'killStreakRate': [np.var,np.max, np.mean],
        'healthItems': [np.var, np.mean],
        'healsOnKill': [ np.var, np.mean],
        'sniper': [ np.var, np.mean],
        'totalDistance': [sum, np.var, np.mean],
        'totalDistancePen': [ sum ,np.var, np.mean],
        'killsPerRoadDist': [np.mean],
        'killsPerWalkDist': [np.mean],
        'killsPerDist': [np.mean],
        'distance_over_weapons': [np.mean],
        'walkDistance_over_heals': [np.mean],
        'skill': [np.var,np.mean]
}

**ميزات جديدة** <br/>
<br/>      `killPlaceScall` - ميزة متدرجة `killPlace`. ما عليك سوى تقسيم `killPlace` على عدد اللاعبين في المباراة.
<br/>      `damageRate` - النسبة `kills` و`damageDealt/100`. إذا `damageRate`>1، قام اللاعب بقتل الأعداء الذين تضرروا بالفعل. لذلك كان قتلهم أسهل.
إذا كانت هذه الميزة <1، فهذا يعني أن اللاعب يتسبب في ضرر أكبر مما يقتله - خاض اللاعب معركة صعبة أو ألحق ضررًا بسيطًا ببعض اللاعبين الذين لم يقتلهم. 
<br/>     `headshotRate` - ​​النسبة المئوية لعمليات القتل بالرصاص. يظهر مهارة اللاعب
<br/>     `killStreakRate` - النسبة المئوية لسلسلة القتل من جميع عمليات القتل. يظهر أيضًا مهارة اللاعب
<br/>     `healthItems` - إجمالي عدد العناصر الصحية (الشفاء+التعزيزات). 
<br/>     `healsOnKill` - يساوي `healsItems`/`kills`. إنه يوضح مدى جودة اللاعب في المعركة. إذا لم يستخدم اللاعب عمليات الشفاء بعد القتل، فهذا يعني على الأرجح أنه لن يتعرض للضرر.
<br/>     `sniper` - يساوي `longestKill`/100*`weaponsAcquired`. يظهر مهارة القناص للاعب. عادة ما يكون لدى القناصين أسلحة جيدة. للعثور على هذا السلاح، يحتاج اللاعب الأكثر احتمالاً إلى الحصول على الكثير من الأسلحة الأخرى. نعم إنها خاصية غريبة
<br/>     `totalDistance` - `rideDistance` + `walkDistance` + `swimDistance`.  المسافة الكبيرة تعني أن اللاعب بقي على قيد الحياة لفترة طويلة من الزمن، لذلك سيحصل على مركز نهائي جيد.
<br/>     `totalDistancePen` - تمت معاقبته `totalDistance`. من الضروري التنبؤ بوقت مباراة اللاعب. لذا فإن سرعة السيارة أعلى بحوالي 5 مرات من سرعة مشي اللاعب وسرعة السباحة أقل بحوالي 10 مراتr من سرعة مشي اللاعب.
<br/>     `killsPerRoadDist` - يقتل لكل مسافة. يمكن لهذه الميزة إظهار مهارتك أيضًا. من الصعب قتل العدو باستخدام السيارة. 
<br/>     `killsPerWalkDist` - يمثل أسلوب اللاعب. إنه يظهر أنك في العربة أو في حالة تحرك دائم.
<br/>     `killsPerDist` - فقط مزيج من `killsPerRoadDist` و`killsPerWalkDist`
<br/>     `distance_over_weapons` - يمكن أن تشير القيم المنخفضة إلى أن اللاعب يحاول العثور على الغنائم بنفسه و/أو أنه/هي لا يفي بمعداته/معداتها، وقد تعني القيم العالية أن اللاعب يأخذ الغنائم من الأشخاص المقتولين و/أو لديه/لديها معدات جيدة. بالطبع، هذا ليس صحيحًا دائمًا.
<br/>     `walkDistance_over_heals` - قد يمثل معارك اللاعبين لكل مسافة.
<br/>     `skill` - يساوي `headshotKills` + `roadKills` + `teamKills`. مجرد واحد من مؤشرات مهارة اللاعب.


In [ ]:
def featuring(df, isTrain, Debug):
    y=None
    if Debug:
        df = df[df['matchId'].isin(df['matchId'].unique()[:2000])]
 
    #Creating new features
    #_________________________________________________________________________________________

    nplayers = df.groupby('matchId')['matchId'].transform('count')
    df['killPlaceScall'] = df['killPlace'] / nplayers
    df['damageRate'] = df['kills']/(0.01*df['damageDealt'])
    df['headshotRate'] = df['headshotKills']/df['kills']
    df['killStreakRate'] = df['killStreaks']/df['kills']
    df['healthItems'] = df['heals'] + df['boosts']
    df['healsOnKill'] = df['healthItems']/df['kills']
    df['sniper'] = df['longestKill']/100*df['weaponsAcquired']
    df['totalDistance'] = df['rideDistance'] + df["walkDistance"] + df["swimDistance"]
    df['totalDistancePen'] = df['rideDistance']/5 + df["walkDistance"] + df["swimDistance"]*10
    df['killsPerRoadDist'] = df['roadKills'] / (df['rideDistance']+1)
    df['killsPerWalkDist'] = (df['kills']-df['roadKills']) / (df['walkDistance']+1)
    df['killsPerDist'] = df['kills']/(df['totalDistance']+1)
    df['distance_over_weapons'] = df['totalDistance'] / df['weaponsAcquired']
    df['walkDistance_over_heals'] = df['walkDistance']/100/df['heals']
    df["skill"] = df["headshotKills"] + df["roadKills"] - df['teamKills'] 
    df.fillna(0,inplace=True)
    df.replace(np.inf, 0, inplace=True)
    #_________________________________________________________________________________________
    
    ids = df[['matchId','groupId','Id']]
    df.drop(columns=['killPlace','killPoints','rankPoints','winPoints','matchType','maxPlace','Id'],inplace=True)
    
    tfeatures = team_features.copy()
    if isTrain:
        tfeatures['winPlacePerc'] = max
    X = df.groupby(['matchId','groupId']).agg(tfeatures)
    X.fillna(0,inplace=True)
    X.replace(np.inf, 1000000, inplace=True)
    X = reduce_mem_usage(X)    
    if isTrain:
        y = X[('winPlacePerc','max')]     
        X.drop(columns=[('winPlacePerc','max')],inplace=True)
             
   
    #Group dataset by matches. To each match apply ranking 
    X_ranked = X.groupby('matchId').rank(pct=True)    
    X = X.reset_index()[['matchId','groupId']].merge(X_ranked, suffixes=["", "_rank"], how='left', on=['matchId', 'groupId'] )

    ids_after = X[['matchId','groupId']]
    ids_after.columns = ['matchId','groupId']
    
    X = X.drop(['matchId','groupId'],axis=1)
    X.columns = [a+"_"+b for a,b in X.columns]
    X = reduce_mem_usage(X)
    
    return X, y, ids,ids_after

In [ ]:
%%time
X_train, y, _,_ = featuring(train,True,False)
X_test, _,ids_init,ids_after = featuring(test,False,False)


قم بتقسيم مجموعة بيانات القطار الخاصة بنا مرة أخرى


In [ ]:
X_train, X_holdout, y_train, y_holdout = train_test_split(X_train, y, test_size=0.2, random_state=666)

In [ ]:
%%time
lgtrain = lgb.Dataset(X_train, label=y_train.reset_index(drop=True))
res = lgb.cv({'metric': 'mae'},lgtrain, nfold=5,stratified=False,seed=666)
print("Mean score:",res['l1-mean'][-1])


لقد حصلنا على تحسن كبير (مرتين تقريبًا). لذا فإن الميزات الجديدة تساعد حقًا في فهم البيانات.
<br/>
<br/> 
الآن دعونا نضبط LightGBM. للقيام بذلك، سنستخدم GridSearchCV، مما يساعد في العثور على أفضل المعلمات.


In [ ]:
gridParams = {
    'num_leaves': [30,50,100], 'max_depth': [-1,8,15], 
    'min_data_in_leaf': [100,300,500], 'max_bin': [250,500], 
    'lambda_l1': [0.01], 'num_iterations': [5], 
    'nthread': [4], 'seed': [666],
    'learning_rate': [0.05], 'metric': ['mae'],
    "bagging_fraction" : [0.7], "bagging_seed" : [0], "colsample_bytree" : [0.7]
    }
model = lgb.LGBMRegressor()
grid = GridSearchCV(model, gridParams,
                    verbose=1,
                    cv=5)

سنقوم بضبط `num_leaves` و`max_depth` و`min_data_in_leaf` و`max_bin`، لأنها المعلمات الرئيسية في LightGBM. 
<br/>     `num_leaves` - ​​الحد الأقصى لعدد الأوراق في شجرة واحدة. إنها المعلمة الرئيسية للتحكم في مدى تعقيد نموذج الشجرة.
<br/>     `max_depth` - حدد أقصى عمق لنموذج الشجرة. يستخدم هذا للتعامل مع الإفراط في التركيب. -1 يعني لا يوجد حد
<br/>     `min_data_in_leaf` - الحد الأدنى لعدد البيانات في ورقة واحدة. هذه معلمة مهمة جدًا لمنع الإفراط في تركيبها في شجرة ذات أوراق شجر. وتعتمد قيمته المثالية على عدد عينات التدريب و<br/>     `num_leaves`.
<br/>     `max_bin` - الحد الأقصى لعدد الصناديق التي سيتم تجميع قيم الميزات فيها. قد يؤدي العدد الصغير من الصناديق إلى تقليل دقة التدريب ولكنه قد يزيد من الطاقة العامة (التعامل مع الملاءمة الزائدة)



هناك نأخذ 500000 فريق فقط من أصل 1500000. وكما سنرى أبعد من ذلك (في منحنى التعلم)، يكفي العثور على أفضل المعلمات.


In [ ]:
%%time
grid.fit(X_train.iloc[:500000,:], y_train.iloc[:500000])

In [ ]:
print("Best params:", grid.best_params_)
print("\nBest score:", grid.best_score_)
params = grid.best_params_


أفضل نتيجة هي أسوأ مما كانت عليه بعد التحقق من الصحة، لأنه تم أخذ 5 تكرارات، وفي التحقق من الصحة - 100 تكرار. ولكن سيكون الأمر على ما يرام، عندما نقوم بتعيين عدد أكبر من التكرارات مع المعلمات، وهو ما نجده الآن.



## 10. رسم منحنيات التدريب والتحقق من الصحة



الآن دعونا نرسم منحنى التعلم بأحجام مختلفة من مجموعات القطارات.


In [ ]:
from sklearn.model_selection import validation_curve
from sklearn.model_selection import learning_curve
model = lgb.LGBMRegressor(learning_rate=0.05,nthread=4)

def plot_with_err(x, data, **kwargs):
    mu, std = data.mean(1), data.std(1)
    lines = plt.plot(x, mu, '-', **kwargs)
    plt.fill_between(x, mu - std, mu + std, edgecolor='none',
    facecolor=lines[0].get_color(), alpha=0.2)
    
def plot_learning_curve():
    train_sizes = [1000,5000,10000,50000,100000,500000]
    N_train, val_train, val_test = learning_curve(model,
    X_train, y_train, train_sizes=train_sizes, cv=5,
    scoring='neg_mean_absolute_error')
    plot_with_err(N_train, abs(val_train), label='training scores')
    plot_with_err(N_train, abs(val_test), label='validation scores')
    plt.xlabel('Training Set Size'); plt.ylabel('MAE')
    plt.legend()

plot_learning_curve()

كما نرى، في الأحجام الصغيرة من مجموعات القطارات، لدينا فرق كبير في درجات التدريب والتحقق من الصحة. والسبب هو التجهيز الزائد لمجموعة القطارات ونقص البيانات.
<br/> ولكن مع زيادة الحجم، تتقارب هذه المنحنيات. مع حجم قطار يبلغ 500000، يكون هذا الفارق صغيرًا جدًا. لهذا السبب حصلت على 500000 بدلاً من كل مجموعة القطارات في GridSearchCV.



انظر الآن كيف تعتمد النتيجة على عدد التكرارات.


In [ ]:
def iter_vs_score(num_iterations):
    val_train, val_test = validation_curve(model, X_train[:500000], y_train[:500000],
        'num_iterations', num_iterations, cv=4,scoring='neg_mean_absolute_error', verbose=1)
    plot_with_err(num_iterations, abs(val_test), label='validation scores')
    plot_with_err(num_iterations, abs(val_train), label='training scores')
    plt.xlabel('Number of iterations'); plt.ylabel('MAE')
    plt.legend();
    plt.show();

num_iterations_small = [5,10,20,30,100,200]
iter_vs_score(num_iterations_small)
num_iterations_big = [500,1000,5000,10000]
iter_vs_score(num_iterations_big)


بالنسبة لعدد صغير من التكرارات، يسقط الخطأ بسرعة. بالنسبة للتكرارات الكبيرة، ينخفض ​​الخطأ، ولكن ببطء. يمكننا أيضًا أن نلاحظ أن درجات التحقق والتدريب هي نفسها تقريبًا بالنسبة لعدد صغير من التكرارات. بالنسبة لعدد كبير من التكرارات، يمكننا أن نرى أن النتيجة لمجموعة التدريب تستمر في الانخفاض، وبالنسبة لمجموعة التحقق من الصحة، تنخفض أيضًا، ولكن بشكل أبطأ بكثير. لذا تتباعد المنحنيات، لكن لا يوجد فرط في التجهيز، لأن درجة التحقق تستمر في الانخفاض.



## 11. التنبؤ بالعينات الاختبارية والمحتجزة



دعونا ندرب نموذج LightGBM باستخدام المعلمات، التي وجدناها باستخدام GridSearchCV.  في نفس الوقت سوف نقوم بحساب الخطأ في مجموعة الإيقاف كل 1000 تكرار. إجمالي عدد التكرارات هو 5000، وهذا يكفيني. إذا أخذنا عددًا أكبر من التكرارات، فلن نحصل على تحسن كبير أو حتى قد نحصل على فرط التجهيز.


In [ ]:
%%time
lgtrain = lgb.Dataset(X_train, label=y_train)
lgval = lgb.Dataset(X_holdout, label=y_holdout)

params['num_iterations'] = 5000
model = lgb.train(params, lgtrain, valid_sets=[lgtrain, lgval], early_stopping_rounds=200, verbose_eval=1000)


نحصل على 0.0291 لمجموعة الإيقاف و0.0242 لمجموعة القطار. من الواضح أنها أفضل من نتائجنا السابقة. قم الآن بالتنبؤ بمجموعة الاختبار ووضع النتائج في ملف `submission.csv`


In [ ]:
pred_test = model.predict(X_test, num_iteration=model.best_iteration)

ids_after['winPlacePerc'] = pred_test
predict = ids_init.merge(ids_after, how='left', on=['groupId',"matchId"])['winPlacePerc']
df_sub = pd.read_csv("../input/sample_submission_V2.csv")
df_test = pd.read_csv("../input/test_V2.csv")
df_sub['winPlacePerc'] = predict
df_sub[["Id", "winPlacePerc"]].to_csv("submission.csv", index=False)


في المتصدرين العامة أحصل على 0.0272. ليس سيئا)



## 12. الاستنتاجات


    نحصل على درجة جيدة. ولكن، بالطبع، يمكن أن يكون أفضل. هناك الكثير من الطرق للقيام بذلك. على سبيل المثال، قمت بحذف ميزات `killPoints` و`rankPoints` و`winPoints`. وقد تكون مفيدة، إذا تم تفسيرها بشكل صحيح. هناك أيضًا الكثير من الغشاشين في اللعبة. لذلك يجب معالجة الغشاشين. لقد قمت بضبط LightGBM قليلاً، ويمكننا العثور على معلمات أفضل أو حتى تجربة طراز آخر.
<br/>    في البداية، تحدثت عن [PUBG Developer API](https://developer.pubg.com/). يمكننا الحصول على المزيد من الميزات به، حتى نتمكن من صنع نموذج أكثر تعقيدًا. سيكون من الرائع إنشاء تطبيق يقدم لك النصائح في الوقت الفعلي أثناء المباراة. يمكن لهذا الحل تقريب هذه الفكرة أو مجرد مساعدة مجتمع PUBG بطريقة أخرى.



شكرا لك على القراءة وآسف لغتي الإنجليزية)



<img src="@@KEEP_00064@@ width="600" height="600"> 



افريموف إيفان